# Surya Customer Churn Analytics

## 1. Project Overview

This consolidated notebook presents a Customer Churn Prediction and Retention Analytics project for a bank customer dataset. It preserves the verified work from the original seven notebooks while removing repeated exploratory and checkpoint material. The workflow moves from data understanding and cleaning to feature engineering, model evaluation, explainability, and retention strategy.

Verified project assets include the raw dataset, cleaned and modeling datasets, SQL analysis files, baseline and optimized machine learning reports, saved XGBoost model artifacts, SHAP feature importance output, native XGBoost feature importance output, and retention prioritization tables.


## 2. Business Problem

Customer churn reduces customer lifetime value and increases acquisition pressure. The business problem is to identify customers who are likely to leave and translate model output into practical retention priorities. The project therefore treats churn prediction as both a machine learning task and a business analytics task.


## 3. Objectives

- Understand churn patterns across geography, gender, age, activity, products, balance, tenure, and credit card status.
- Prepare a modeling dataset without identifier leakage.
- Compare baseline and optimized classification models.
- Select a final operational model and classification threshold.
- Explain model behavior using SHAP and native XGBoost feature importance.
- Convert churn probability into customer risk, value, priority, and recommended action segments.
- Clearly distinguish implemented machine learning from future Generative AI enhancement ideas.


## 4. Dataset

The raw dataset contains **10,000 customer records** and **14 columns**. The target variable is `Exited`, where `1` means the customer churned and `0` means the customer stayed.

- Stayed customers: **7,963**
- Churned customers: **2,037**
- Overall churn rate: **20.37%**
- Missing values found in the raw dataset: **0**

The original identifier columns `RowNumber`, `CustomerId`, and `Surname` are useful for record identification but are excluded from model training to avoid leakage or non-generalizable identifiers.


In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw" / "bank_customer_churn.csv"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
MODELS = PROJECT_ROOT / "models"

df = pd.read_csv(DATA_RAW)
df.head()


## 5. Data Understanding

The project inspected the schema, target distribution, missing values, duplicates, and descriptive summaries. The data has no missing values in the verified raw file. The target distribution is imbalanced: about one in five customers churned.


In [ ]:
print(df.shape)
display(df.info())
display(df.describe(include="all").T)
display(df["Exited"].value_counts().rename(index={0: "Stayed", 1: "Churned"}))
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())


## 6. Data Cleaning

The raw file was already complete: no missing values were found. The cleaning step standardizes the modeling frame by removing non-predictive identifiers and retaining customer attributes that can generalize to unseen customers.


In [ ]:
identifier_cols = ["RowNumber", "CustomerId", "Surname"]
target = "Exited"

cleaned_df = df.copy()
model_base = cleaned_df.drop(columns=identifier_cols)
model_base.head()


## 7. Exploratory Data Analysis

The strongest verified churn patterns are summarized below.


### Verified EDA Tables

**Churn by geography**

| Geography | customers | churned | churn_rate |
| --- | --- | --- | --- |
| Germany | 2509 | 814 | 32.44 |
| Spain | 2477 | 413 | 16.67 |
| France | 5014 | 810 | 16.15 |

**Churn by gender**

| Gender | customers | churned | churn_rate |
| --- | --- | --- | --- |
| Female | 4543 | 1139 | 25.07 |
| Male | 5457 | 898 | 16.46 |

**Churn by age group**

| AgeGroup | customers | churned | churn_rate |
| --- | --- | --- | --- |
| 46-55 | 1311 | 663 | 50.57 |
| 56-65 | 536 | 259 | 48.32 |
| 36-45 | 3736 | 733 | 19.62 |
| 66+ | 264 | 35 | 13.26 |
| 26-35 | 3542 | 301 | 8.5 |
| 18-25 | 611 | 46 | 7.53 |

**Churn by activity**

| ActivityStatus | customers | churned | churn_rate |
| --- | --- | --- | --- |
| Inactive | 4849 | 1302 | 26.85 |
| Active | 5151 | 735 | 14.27 |

**Churn by number of products**

| NumOfProducts | customers | churned | churn_rate |
| --- | --- | --- | --- |
| 4 | 60 | 60 | 100.0 |
| 3 | 266 | 220 | 82.71 |
| 1 | 5084 | 1409 | 27.71 |
| 2 | 4590 | 348 | 7.58 |


In [ ]:
eda_tables = {
    "geography": pd.read_csv(DATA_PROCESSED / "churn_by_geography.csv"),
    "gender": pd.read_csv(DATA_PROCESSED / "churn_by_gender.csv"),
    "age_group": pd.read_csv(DATA_PROCESSED / "churn_by_age_group.csv"),
    "activity": pd.read_csv(DATA_PROCESSED / "churn_by_activity.csv"),
    "products": pd.read_csv(DATA_PROCESSED / "churn_by_products.csv"),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()
for ax, (name, table) in zip(axes, eda_tables.items()):
    x_col = table.columns[0]
    sns.barplot(data=table, x=x_col, y="churn_rate", ax=ax, color="#4C78A8")
    ax.set_title(f"Churn rate by {name.replace('_', ' ')}")
    ax.set_ylabel("Churn rate (%)")
    ax.tick_params(axis="x", rotation=30)
axes[-1].axis("off")
plt.tight_layout()
plt.show()


## 8. Feature Engineering

The verified feature engineering created business-readable segments used in both modeling and retention analysis:

- `AgeGroup`
- `BalanceStatus`
- `ActivityStatus`
- `AgeActivity`

The modeling dataset keeps `Exited` as the target and excludes customer identifiers.


In [ ]:
def add_business_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()
    out["AgeGroup"] = pd.cut(
        out["Age"],
        bins=[17, 25, 35, 45, 55, 65, np.inf],
        labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66+"],
    )
    out["BalanceStatus"] = np.where(out["Balance"] > 0, "Positive Balance", "Zero Balance")
    out["ActivityStatus"] = np.where(out["IsActiveMember"] == 1, "Active", "Inactive")
    out["AgeActivity"] = out["AgeGroup"].astype(str) + "_" + out["ActivityStatus"]
    return out

model_df = add_business_features(cleaned_df).drop(columns=identifier_cols)
model_df.head()


## 9. Model Preparation

The original project used an 80/20 stratified train-test split with `random_state=42`. Preprocessing was fit only on the training data and applied to the test data, which avoids fitting transformations on the full dataset. Numeric variables were scaled and categorical variables were one-hot encoded.


In [ ]:
X = model_df.drop(columns=[target])
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
X_train.shape, X_test.shape


## 10. Baseline Models

The baseline stage compared a naive majority-class baseline, Logistic Regression, Random Forest, and XGBoost.


| Model | Accuracy | Precision | Recall | F1 | ROC_AUC | PR_AUC |
| --- | --- | --- | --- | --- | --- | --- |
| Naive Baseline | 0.7965 | 0.0 | 0.0 | 0.0 |  |  |
| Logistic Regression | 0.7845 | 0.4806451612903226 | 0.7321867321867321 | 0.5803310613437196 | 0.850580935326698 | 0.6945529198878357 |
| Random Forest | 0.843 | 0.6131386861313869 | 0.6191646191646192 | 0.6161369193154034 | 0.8475609662050341 | 0.6576624124062486 |
| XGBoost | 0.8695 | 0.7786259541984732 | 0.5012285012285013 | 0.6098654708520179 | 0.8653723060502723 | 0.7162431553182309 |


In [ ]:
baseline_results = pd.read_csv(REPORTS / "baseline_model_results.csv")
display(baseline_results)


## 11. Model Optimization

The optimization stage tuned Logistic Regression, Random Forest, and XGBoost. XGBoost was selected as the final model because it had the strongest overall optimized result in the saved reports.


| Model | Accuracy | Precision | Recall | F1 | ROC_AUC | PR_AUC |
| --- | --- | --- | --- | --- | --- | --- |
| Optimized Logistic Regression | 0.785 | 0.4813614262560778 | 0.7297297297297297 | 0.580078125 | 0.8505115284776302 | 0.6952023201461549 |
| Optimized Random Forest | 0.815 | 0.5339449541284403 | 0.714987714987715 | 0.6113445378151261 | 0.8618634042362855 | 0.6908220981322807 |
| Optimized XGBoost | 0.872 | 0.7849056603773585 | 0.5110565110565111 | 0.6190476190476191 | 0.8631050156473885 | 0.7138869362100294 |


In [ ]:
optimized_results = pd.read_csv(REPORTS / "optimized_model_results.csv")
display(optimized_results)


## 12. Final XGBoost Model

The saved `model_config.json` identifies the final model as **XGBoost (Optimized)** with operational classification threshold **0.35**. Saved model artifacts include `final_churn_model.joblib` and `final_preprocessor.joblib`.


In [ ]:
with open(MODELS / "model_config.json", "r", encoding="utf-8") as f:
    model_config = json.load(f)

final_model = joblib.load(MODELS / "final_churn_model.joblib")
final_preprocessor = joblib.load(MODELS / "final_preprocessor.joblib")
model_config


## 13. Model Evaluation

At the saved operational threshold of **0.35**, the final report metrics are:

- Accuracy: **0.8540**
- Precision: **0.6501**
- Recall: **0.6118**
- F1: **0.6304**
- ROC-AUC: **0.8631**
- PR-AUC: **0.7139**


In [ ]:
final_metrics = pd.read_csv(REPORTS / "final_model_metrics.csv")
display(final_metrics)


## 14. Threshold Comparison

The project configuration uses threshold **0.35**. The saved threshold analysis shows that:

- Threshold **0.35**: precision **0.6501**, recall **0.6118**, F1 **0.6304**
- Threshold **0.50**: precision **0.7849**, recall **0.5111**, F1 **0.6190**

This corrects the earlier audit note: the verified project files show that `0.35` gives higher recall and F1, while `0.50` gives higher precision and the higher reported accuracy in the optimized model results. The better threshold depends on the cost of false positives versus false negatives. For retention use cases, a lower threshold can be reasonable because missing a true churn-risk customer may be more costly than contacting an extra at-risk customer.


In [ ]:
threshold_df = pd.read_csv(REPORTS / "optimized_xgboost_threshold_analysis.csv")
display(threshold_df)

plt.figure(figsize=(8, 5))
for metric in ["Precision", "Recall", "F1"]:
    plt.plot(threshold_df["Threshold"], threshold_df[metric], marker="o", label=metric)
plt.axvline(0.35, color="#D62728", linestyle="--", label="Operational threshold 0.35")
plt.axvline(0.50, color="#2CA02C", linestyle=":", label="Default threshold 0.50")
plt.xlabel("Classification threshold")
plt.ylabel("Score")
plt.title("Optimized XGBoost threshold comparison")
plt.legend()
plt.tight_layout()
plt.show()


## 15. SHAP Explainability

SHAP mean absolute importance and native XGBoost feature importance do not produce identical rankings because they measure importance differently. SHAP summarizes average contribution to model output, while native XGBoost importance reflects tree-splitting behavior. Both are retained because together they provide a fuller explanation.


**Top SHAP features**

| Feature | MeanAbsSHAP | OriginalFeature |
| --- | --- | --- |
| cat__NumOfProducts_2 | 0.7935903 | NumOfProducts |
| num__Age | 0.69575727 | Age |
| cat__Geography_Germany | 0.26191524 | Geography |
| cat__Gender_Male | 0.25261644 | Gender |
| cat__ActivityStatus_Inactive | 0.24193949 | ActivityStatus |
| num__Balance | 0.18648377 | Balance |
| num__CreditScore | 0.16680296 | CreditScore |
| cat__BalanceStatus_Zero Balance | 0.16450739 | BalanceStatus |
| cat__NumOfProducts_3 | 0.16135292 | NumOfProducts |
| num__EstimatedSalary | 0.1551979 | EstimatedSalary |

**Top native XGBoost importance features**

| Feature | Importance | OriginalFeature |
| --- | --- | --- |
| cat__AgeActivity_46-55_Inactive | 0.28002715 | AgeActivity |
| cat__NumOfProducts_2 | 0.09529173 | NumOfProducts |
| remainder__IsActiveMember | 0.07001885 | IsActiveMember |
| cat__AgeActivity_56-65_Inactive | 0.063895665 | AgeActivity |
| cat__NumOfProducts_3 | 0.05983074 | NumOfProducts |
| cat__AgeGroup_26-35 | 0.04698779 | AgeGroup |
| cat__AgeActivity_66+_Active | 0.04155505 | AgeActivity |
| num__Age | 0.038779356 | Age |
| cat__ActivityStatus_Inactive | 0.03723757 | ActivityStatus |
| cat__BalanceStatus_Zero Balance | 0.031218538 | BalanceStatus |


In [ ]:
shap_importance = pd.read_csv(REPORTS / "shap_feature_importance.csv")
xgb_importance = pd.read_csv(REPORTS / "xgboost_feature_importance.csv")
display(shap_importance.head(10))
display(xgb_importance.head(10))


## 16. Customer Risk Analysis

The retention analysis transforms churn probability into `RiskScore`, `RiskLevel`, `CustomerValueScore`, `CustomerValueSegment`, `RetentionPriority`, and `RecommendedAction`. This step connects model output to business action.


**Retention priority summary**

| RetentionPriority | Customers | Percentage |
| --- | --- | --- |
| Monitor | 7616 | 76.16000000000001 |
| Medium | 649 | 6.49 |
| High | 1009 | 10.09 |
| Critical | 726 | 7.26 |

**Risk and value matrix**

| RiskLevel | Low Value | Medium Value | High Value |
| --- | --- | --- | --- |
| Low | 3006 | 1796 | 2152 |
| Moderate | 259 | 403 | 455 |
| High | 194 | 373 | 308 |
| Very High | 217 | 419 | 418 |


In [ ]:
retention = pd.read_csv(REPORTS / "retention_strategy.csv")
priority_summary = pd.read_csv(REPORTS / "retention_priority_summary.csv")
risk_value_matrix = pd.read_csv(REPORTS / "risk_value_matrix.csv")
display(priority_summary)
display(risk_value_matrix)
display(retention.head())


## 17. Retention Strategy

Retention actions are prioritized by risk and customer value. Critical customers are high-risk customers who require priority outreach, while monitor customers should receive regular engagement rather than expensive intervention.


**Priority by geography**

| Geography | Critical | High | Medium | Monitor |
| --- | --- | --- | --- | --- |
| France | 193 | 356 | 272 | 4193 |
| Germany | 425 | 469 | 241 | 1374 |
| Spain | 108 | 184 | 136 | 2049 |

**Priority by activity**

| ActivityStatus | Critical | High | Medium | Monitor |
| --- | --- | --- | --- | --- |
| Active | 224 | 284 | 314 | 4329 |
| Inactive | 502 | 725 | 335 | 3287 |

**Priority by products**

| NumOfProducts | Critical | High | Medium | Monitor |
| --- | --- | --- | --- | --- |
| 1 | 419 | 846 | 498 | 3321 |
| 2 | 117 | 40 | 150 | 4283 |
| 3 | 149 | 104 | 1 | 12 |
| 4 | 41 | 19 | 0 | 0 |


In [ ]:
display(pd.read_csv(REPORTS / "priority_by_geography.csv"))
display(pd.read_csv(REPORTS / "priority_by_activity.csv"))
display(pd.read_csv(REPORTS / "priority_by_products.csv"))


## 18. Business Insights

- Germany has the highest verified churn rate at 32.44%, compared with Spain at 16.67% and France at 16.15%.
- Female customers have a higher churn rate than male customers in the dataset.
- Customers aged 46-55 and 56-65 have the highest churn rates.
- Inactive customers churn more often than active customers.
- Customers with three or four products have very high churn rates, suggesting product mix or service fit issues.
- SHAP highlights number of products, age, Germany, gender, activity, balance, and credit score as influential model signals.


## 19. AI Enhancement

### Implemented

The implemented project contains machine learning and explainable AI techniques: Logistic Regression, Random Forest, XGBoost, threshold tuning, SHAP explainability, model-based customer risk scoring, and retention priority segmentation.

### Future AI Enhancement

The files do not prove that Generative AI was implemented. Future AI enhancements could include natural-language churn insight summaries, automated customer risk narratives, personalized retention recommendation text, natural-language querying over churn KPIs, and automated insight generation for managers. XGBoost is not Generative AI; it is a supervised machine learning model.


## 20. Limitations

- The PBIX dashboard exists, but dashboard contents were not programmatically verified in this environment.
- The dataset appears to be a static historical dataset, so deployment behavior and live drift were not tested.
- The threshold decision depends on business costs that are not directly quantified in the project files.
- Retention value scoring is a practical proxy rather than a validated customer lifetime value model.


## 21. Future Scope

- Validate the model on newer customer cohorts.
- Add cost-sensitive threshold selection using actual retention campaign economics.
- Monitor model drift after deployment.
- Integrate with CRM workflows for controlled outreach experiments.
- Extend Power BI reporting after verifying dashboard pages and visuals.
- Add future Generative AI assistance for natural-language explanations and recommendation drafting.


## 22. Conclusion

The final selected model is optimized XGBoost with an operational threshold of **0.35**. The verified operational metrics at threshold 0.35 are accuracy **0.8540**, precision **0.6501**, recall **0.6118**, F1 **0.6304**, ROC-AUC **0.8631**, and PR-AUC **0.7139**. The project connects predictive modeling with explainability and retention actions, making it suitable as a Data Analytics with AI internship submission while accurately identifying Generative AI as future scope rather than implemented work.
